<a href="https://colab.research.google.com/github/ClaretWheel1481/Artificial-Intelligence/blob/main/DeepLearning/%E8%A7%86%E9%A2%91%E5%88%86%E7%B1%BB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os  # 用于文件和目录操作
import cv2  # 用于读取和处理视频帧 pip install opencv-python
from PIL import Image  # 用于图像处理，转换为 PIL.Image 格式
import torch  # 用于深度学习模型的实现
import torch.nn as nn  # 用于构建神经网络层
import torch.optim as optim  # 用于优化算法
from torch.utils.data import Dataset, DataLoader  # 用于自定义数据集和数据加载
from torchvision import transforms  # 用于图像的预处理和变换

In [ ]:
# 定义视频数据集类
class VideoDataset(Dataset):
    def __init__(self, video_dir, seq_length, transform=None):
        """
        video_dir: 视频文件夹路径
        seq_length: 每个样本的帧数
        transform: 图像的预处理方法
        """
        self.video_dir = video_dir  # 保存视频文件夹路径
        self.seq_length = seq_length  # 每个视频样本的帧数
        self.transform = transform  # 图像预处理
        # 获取所有 mp4 格式的视频文件路径
        self.video_files = [os.path.join(video_dir, f) for f in os.listdir(video_dir) if f.endswith('.mp4')]

    def __len__(self):
        """返回数据集中视频的数量"""
        return len(self.video_files)

    def __getitem__(self, idx):
        """根据索引返回视频样本"""
        video_path = self.video_files[idx]  # 获取当前视频的路径
        frames = self._read_video_frames(video_path)  # 读取视频的帧
        label = idx % 3  # 假设每个视频的标签为 (idx % 3)，模拟一个3分类问题

        # 如果定义了 transform，对每一帧应用 transform
        if self.transform:
            frames = torch.stack([self.transform(frame) for frame in frames])

        return frames, label  # 返回处理后的帧和标签

    def _read_video_frames(self, video_path):
        """
        读取视频文件，提取等间隔的帧。
        """
        cap = cv2.VideoCapture(video_path)  # 使用 OpenCV 打开视频文件
        frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))  # 获取视频的总帧数
        frame_indices = torch.linspace(0, frame_count - 1, self.seq_length).long()  # 选择均匀间隔的帧
        frames = []  # 存储视频帧

        for idx in range(frame_count):
            ret, frame = cap.read()  # 读取视频帧
            if not ret:
                break  # 如果帧读取失败，跳出循环
            if idx in frame_indices:  # 如果当前帧在选择的帧索引列表中
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)  # 转换为 RGB 格式
                frame = cv2.resize(frame, (128, 128))  # 将帧调整为 128x128 大小
                frame = Image.fromarray(frame)  # 转换为 PIL.Image 格式
                frames.append(frame)  # 将当前帧加入到帧列表

        cap.release()  # 释放视频资源
        return frames  # 返回帧列表

In [ ]:
# 定义 CNN-RNN 模型类
class CNNRNNModel(nn.Module):
    def __init__(self, rnn_hidden_size, output_size):
        super(CNNRNNModel, self).__init__()

        # CNN 模块
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, stride=1, padding=1),  # 第一个卷积层，输入 3 个通道，输出 16 个通道
            nn.ReLU(),  # 激活函数 ReLU
            nn.MaxPool2d(kernel_size=2, stride=2),  # 最大池化层，减少尺寸
            nn.Conv2d(16, 32, kernel_size=3, stride=1, padding=1),  # 第二个卷积层，输入 16 个通道，输出 32 个通道
            nn.ReLU(),  # 激活函数 ReLU
            nn.MaxPool2d(kernel_size=2, stride=2),  # 最大池化层，进一步减少尺寸
        )

        # RNN 模块
        self.rnn_hidden_size = rnn_hidden_size  # 设置 RNN 隐藏层的大小
        self.rnn = None  # 延迟初始化 RNN 模块

        # 输出层
        self.fc = nn.Linear(rnn_hidden_size, output_size)  # 全连接层，输出分类结果

    def forward(self, x):
        """
        模型的前向传播过程
        x: 输入的帧序列，形状为 (batch_size, seq_length, channels, height, width)
        """
        batch_size, seq_length, channels, height, width = x.size()  # 获取输入的尺寸

        # 初始化 CNN 特征存储列表
        cnn_features = []

        for t in range(seq_length):
            frame = x[:, t, :, :, :]  # 提取当前帧
            feature = self.cnn(frame)  # 使用 CNN 提取当前帧的特征
            feature = feature.view(batch_size, -1)  # 展平特征为一维向量

            # 动态初始化 RNN
            if self.rnn is None:
                self._initialize_rnn(feature.size(1))  # 如果 RNN 未初始化，初始化 RNN

            cnn_features.append(feature)  # 将当前帧的特征加入到特征列表

        cnn_features = torch.stack(cnn_features, dim=1)  # 将特征堆叠为 (batch_size, seq_length, cnn_feature_size)

        # RNN 处理提取的 CNN 特征
        rnn_out, _ = self.rnn(cnn_features)  # 获取 RNN 输出
        last_output = rnn_out[:, -1, :]  # 获取最后一个时间步的输出
        output = self.fc(last_output)  # 将最后一个时间步的输出传递给全连接层，得到最终分类结果
        return output

    def _initialize_rnn(self, cnn_feature_size):
        """动态初始化 RNN 模块"""
        self.rnn = nn.LSTM(input_size=cnn_feature_size, hidden_size=self.rnn_hidden_size, batch_first=True)

In [ ]:
# 设置超参数
rnn_hidden_size = 128  # RNN 隐藏层大小
output_size = 3  # 输出类别数，这里假设为 3 类
batch_size = 16  # 每批次的样本数
seq_length = 16  # 每个视频样本的帧数
num_epochs = 20  # 训练的轮数
video_dir = "./drive/MyDrive/数据集/video_dataset/videos"  # 视频文件夹路径

In [ ]:
# 定义图像预处理变换和数据集
transform = transforms.Compose([
    transforms.ToTensor(),  # 转换为张量
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # 归一化处理
])
dataset = VideoDataset(video_dir, seq_length, transform)  # 创建视频数据集实例
data_loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)  # 创建数据加载器

In [ ]:
# 初始化模型
model = CNNRNNModel(rnn_hidden_size=rnn_hidden_size, output_size=output_size)
# model.cuda()

In [ ]:
# 定义损失函数和优化器
criterion = nn.CrossEntropyLoss()  # 使用交叉熵损失函数
optimizer = optim.Adam(model.parameters(), lr=0.003)  # 使用 Adam 优化器

In [ ]:
# 训练循环
for epoch in range(num_epochs):
    total_loss = 0.0  # 初始化总损失
    for videos, labels in data_loader:
        # 将视频和标签移动到 CPU 上（如果使用 GPU，请取消注释下面的代码）
        # videos, labels = videos.cuda(), labels.cuda()
        videos, labels = videos.to("cpu"), labels.to("cpu")

        # 前向传播
        outputs = model(videos)  # 模型预测
        print(f"outputs: {outputs}")  # 打印输出结果
        print(f"labels: {labels}")  # 打印标签
        loss = criterion(outputs, labels)  # 计算损失

        # 反向传播和优化
        optimizer.zero_grad()  # 清除梯度
        loss.backward()  # 计算梯度
        optimizer.step()  # 更新模型参数

        total_loss += loss.item()  # 累加损失

    # 打印每轮的平均损失
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {total_loss / len(data_loader):.4f}")

outputs: tensor([[-0.1545, -0.1661, -0.1560],
        [-0.1833, -0.0321, -0.0937],
        [-0.0849,  0.0968,  0.0466],
        [-0.0651, -0.1068, -0.1249],
        [-0.1594, -0.0786, -0.1185],
        [-0.1449, -0.0370, -0.0271],
        [-0.2391, -0.0172,  0.1559],
        [-0.1188,  0.0816, -0.0937]], grad_fn=<AddmmBackward0>)
labels: tensor([0, 0, 2, 1, 1, 2, 1, 0])
Epoch 1/20, Loss: 1.1057
outputs: tensor([[-0.0771,  0.0629, -0.2420],
        [-0.0669, -0.1101, -0.2524],
        [-0.0133, -0.2351, -0.2428],
        [-0.2041,  0.0189,  0.1026],
        [-0.1026,  0.1213, -0.3034],
        [-0.2687,  0.1772,  0.0727],
        [-0.1276,  0.1192, -0.2706],
        [-0.2348,  0.0308,  0.1454]], grad_fn=<AddmmBackward0>)
labels: tensor([0, 0, 0, 2, 1, 1, 1, 2])
Epoch 2/20, Loss: 0.9662
outputs: tensor([[ 0.0686, -0.1163, -0.3873],
        [-0.2129,  0.0306,  0.1683],
        [-0.2572,  0.3187,  0.0721],
        [-0.3029,  0.1344,  0.3036],
        [-0.0706,  0.2816, -0.4020],
        [ 

In [ ]:
# 保存训练好的模型
torch.save(model.state_dict(), "cnn_rnn_model.pth")  # 保存模型的参数

In [ ]:
# 验证模型
model2 = CNNRNNModel(rnn_hidden_size=rnn_hidden_size, output_size=output_size)
model2.load_state_dict(torch.load("cnn_rnn_model.pth"),strict=False)
model2.eval()  # 设置为评估模式
def classify_video(video_path, model, transform, seq_length=16):
    """
    对单个视频进行分类
    Args:
        video_path (str): 视频文件路径
        model (nn.Module): 训练好的 CNN-RNN 模型
        transform (callable): 图像预处理方法
        seq_length (int): 使用的帧数
    Returns:
        int: 分类类别索引
    """
    model.eval()

    # 读取视频并提取帧
    cap = cv2.VideoCapture(video_path)
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_indices = torch.linspace(0, frame_count - 1, seq_length).long()
    frames = []

    for idx in range(frame_count):
        ret, frame = cap.read()
        if not ret:
            break
        if idx in frame_indices:
            # 转换为 RGB 并调整尺寸
            frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frame = cv2.resize(frame, (128, 128))  # 调整为训练时的输入大小
            frame = Image.fromarray(frame)  # 转换为 PIL.Image
            frames.append(transform(frame))

    cap.release()

    if len(frames) < seq_length:
        print("视频帧数不足，无法分类。")
        return None

    # 合并帧序列并添加批量维度
    frames = torch.stack(frames).unsqueeze(0)  # (1, seq_length, channels, height, width)
    # frames = frames.cuda()
    # 模型推理
    with torch.no_grad():
        outputs = model(frames)
        _, predicted = torch.max(outputs, 1)

    return predicted.item()  # 返回类别索引
def classify_videos_in_directory(video_dir, model, transform, seq_length=16):
    """
    对目录中的所有视频进行分类
    Args:
        video_dir (str): 视频目录路径
        model (nn.Module): 训练好的模型
        transform (callable): 图像预处理方法
        seq_length (int): 使用的帧数
    Returns:
        dict: 每个视频的分类结果
    """
    results = {}
    video_files = [os.path.join(video_dir, f) for f in os.listdir(video_dir) if f.endswith('.mp4')]

    for video_path in video_files:
        print(f"正在处理: {video_path}")
        label = classify_video(video_path, model, transform, seq_length)
        results[os.path.basename(video_path)] = label

    return results
# 定义预处理方法
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

# 视频目录路径
video_dir = "./drive/MyDrive/数据集/video_dataset/test-videos"

# 对目录中的所有视频进行分类
results = classify_videos_in_directory(video_dir, model, transform, seq_length=16)
print("分类结果:", results)

<ipython-input-29-1cbc30403eea>:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model2.load_state_dict(torch.load("cnn_rnn_model.pth"),strict=False)


正在处理: ./drive/MyDrive/数据集/video_dataset/test-videos/t1.mp4
正在处理: ./drive/MyDrive/数据集/video_dataset/test-videos/t3.mp4
正在处理: ./drive/MyDrive/数据集/video_dataset/test-videos/t2.mp4
正在处理: ./drive/MyDrive/数据集/video_dataset/test-videos/10.mp4
视频帧数不足，无法分类。
正在处理: ./drive/MyDrive/数据集/video_dataset/test-videos/t4.mp4
正在处理: ./drive/MyDrive/数据集/video_dataset/test-videos/11.mp4
正在处理: ./drive/MyDrive/数据集/video_dataset/test-videos/12.mp4
正在处理: ./drive/MyDrive/数据集/video_dataset/test-videos/13.mp4
正在处理: ./drive/MyDrive/数据集/video_dataset/test-videos/14.mp4
视频帧数不足，无法分类。
正在处理: ./drive/MyDrive/数据集/video_dataset/test-videos/9.mp4
分类结果: {'t1.mp4': 1, 't3.mp4': 0, 't2.mp4': 2, '10.mp4': None, 't4.mp4': 1, '11.mp4': 0, '12.mp4': 0, '13.mp4': 0, '14.mp4': None, '9.mp4': 0}
